In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# ---------------- Helper functions ----------------
def compute_n_sqr_charge(O_charge, psi):
    """Expectation value of plaquette operator in charge basis."""
    return np.vdot(psi, O_charge @ psi).real

def compute_theoretical_plaquette_charge(O_charge, psi):
    """Expectation value of plaquette operator in charge basis."""
    return np.vdot(psi, O_charge @ psi).real

# ---------------- Parameters ----------------
Ns = np.arange(5, 33, 4)
λs_over_gs_0 = np.logspace(-1, 3, num=40, endpoint=True)

mneq0 = False  # Set to False for m=0 case
if mneq0:
    λ_m = 1.0  # Fixed magnetic coupling

# ---------------- Storage ----------------
results_n_sqr_charge_0 = np.zeros((len(Ns), len(λs_over_gs_0)))
results_Pop_charge_0 = np.zeros((len(Ns), len(λs_over_gs_0)))

# ---------------- Loop ----------------
for i, n in enumerate(Ns):
    print(f"N = {n}")
    for j, lamog in enumerate(λs_over_gs_0):
        print(rf"  \lambda / g = {lamog}")

        if mneq0:
            H_charge = inf_space_LGT_plaquette_charge_basis_Matrix_reduced4vars_alphas0(
                EC_g1=1/(4*lamog), EJ1=1.0,
                EC_g2=1/(4*lamog), EJ2=1.0,
                EC_g3=1/(4*lamog), EJ3=1.0,
                EC_g4=1/(4*lamog), EJ4=1.0,
                Nsize=n,
                EC_m1=1/(4*λ_m),EC_m2=1/(4*λ_m),EC_m3=1/(4*λ_m),EC_m4=1/(4*λ_m)
            )
        else:
            H_charge = inf_space_LGT_plaquette_charge_basis_Matrix_reduced4vars_alphas0(
                EC_g1=1/(4*lamog), EJ1=1.0,
                EC_g2=1/(4*lamog), EJ2=1.0,
                EC_g3=1/(4*lamog), EJ3=1.0,
                EC_g4=1/(4*lamog), EJ4=1.0,
                Nsize=n
            )
        O_n_sqr_charge = make_n12_operator_charge_basis_Matrix_reduced4vars_alphas0(n)
        O_Pop_charge = make_plaquette_operator_charge_basis_Matrix_reduced4vars_alphas0(n)
        E_charge, ψ_charge = spla.eigsh(H_charge, k=1, which="SA", tol=1e-10, maxiter=2000)
        ψ0_charge = ψ_charge[:, 0]
        results_n_sqr_charge_0[i, j] = compute_n_sqr_charge(O_n_sqr_charge, ψ0_charge)
        results_Pop_charge_0[i, j] = compute_theoretical_plaquette_charge(O_Pop_charge, ψ0_charge)

In [ ]:
# ---------------- Helper function for analytic expvals in the m->0 limit ----------------
def expvals_operators(q, N=100001):

    theta = np.linspace(-np.pi, np.pi, N) # in radians
    space_theta = XSpace(N, a=-np.pi, b=np.pi)
    x_deg = (theta / 2.0) * (180.0/np.pi)  # Convert to in degrees for mathieu_cem
    x_rad = theta / 2.0

    psi, _ = mathieu_cem(0, q, x_deg)
    norm = np.sqrt(simpson(np.abs(psi)**2, theta))
    psi /= norm

    ntheta1_sqr = space_theta.make_n_power(order=2)
    op_exp_p = np.exp(1j * theta)
    op_exp_m = np.exp(-1j * theta)

    integrand = np.conjugate(psi) * ntheta1_sqr(psi)
    exp_ntheta1_sqr = simpson(integrand.real, theta)
    exp_p = simpson(np.conj(psi) * op_exp_p * psi, theta)
    exp_m = simpson(np.conj(psi) * op_exp_m * psi, theta)

    return exp_ntheta1_sqr.real, exp_p, exp_m

# ---------------- Analytic expvals in the m->0 limit ----------------
q_values = -2.0 * λs_over_gs_0
results_nsqr = []
results_Pop = []

for q in q_values:
    exp_ntheta1_sqr, exp_p, exp_m = expvals_operators(q)

    results_nsqr.append(exp_ntheta1_sqr)
    val = (0.5 * (exp_p * exp_p * exp_m * exp_m + exp_m * exp_m * exp_p * exp_p)).real
    results_Pop.append(val)

results_nsqr = np.array(results_nsqr)
results_Pop = np.array(results_Pop)

In [ ]:
# ---------------- Save results ----------------
if mneq0:
    filename_mneq0 = f"LGT_SCQs_plaquette_reduced_def_OCT25/LGT_SCQs_plaquette_reduced4vars_alphas0_plaquetteop_nsqr_charge_m_neq_0_comparison_λ_m_{λ_m}_draft.npz"
    np.savez_compressed(filename_mneq0,
                        Ns_mneq0=Ns,
                        λs_over_gs_0_mneq0=λs_over_gs_0,
                        results_n_sqr_charge_0_mneq0=results_n_sqr_charge_0,
                        results_Pop_charge_0_mneq0=results_Pop_charge_0,
                        q_values_mneq0=q_values,
                        results_nsqr_mneq0=results_nsqr,
                        results_Pop_mneq0=results_Pop
    )
    print(f"\nResults saved to {filename_mneq0}")
else:
    filename_m0 = f"LGT_SCQs_plaquette_reduced_def_OCT25/LGT_SCQs_plaquette_reduced4vars_alphas0_plaquetteop_nsqr_charge_m0comparison_draft.npz"
    np.savez_compressed(filename_m0,
                        Ns_m0=Ns,
                        λs_over_gs_0_m0=λs_over_gs_0,
                        results_n_sqr_charge_0_m0=results_n_sqr_charge_0,
                        results_Pop_charge_0_m0=results_Pop_charge_0,
                        q_values_m0=q_values,
                        results_nsqr_m0=results_nsqr,
                        results_Pop_m0=results_Pop
    )
    print(f"\nResults saved to {filename_m0}")